# 02 — RFM Customer Segmentation
**Business question this notebook answers:** Who are our highest-value customers, and how should they be segmented for targeted action?

**Method:** Recency (days since last purchase), Frequency (number of distinct orders), Monetary (total spend) — each scored 1–5 by quintile, then combined into named, actionable segments.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
df = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv", parse_dates=["InvoiceDate"])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


## 1. Compute Recency, Frequency, Monetary per customer

In [2]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
).reset_index()

rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,3018.616737
std,1715.572666,209.338707,13.009406,14737.731040
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,348.762500
50%,15314.500000,96.000000,3.000000,898.915000
75%,16797.750000,380.000000,7.000000,2307.090000
max,18287.000000,739.000000,398.000000,608821.650000


## 2. Score each dimension 1–5

Recency is scored in reverse (5 = most recent). `duplicates="drop"` handles cases where too many customers
share the exact same value to form 5 clean quintiles — normal with real transactional data.

In [3]:
rfm["R_score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1], duplicates="drop").astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5], duplicates="drop").astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5], duplicates="drop").astype(int)
rfm["RFM_Score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
0,12346,326,12,77556.46,2,5,5,255
1,12347,2,8,5633.32,5,4,5,545
2,12348,75,5,2019.40,3,4,4,344
3,12349,19,4,4428.69,5,3,5,535
4,12350,310,1,334.40,2,1,2,212


## 3. Map scores to business-readable segments

This is the step that makes RFM useful to a non-technical stakeholder: nobody acts on "455", people act on "Champions" or "At Risk".

In [4]:
def segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost"
    else:
        return "Needs Attention"

rfm["Segment"] = rfm.apply(segment, axis=1)
rfm["Segment"].value_counts()

Segment
Lost               1523
Loyal Customers    1403
Champions          1300
At Risk             824
New Customers       443
Needs Attention     385
Name: count, dtype: int64

## 4. Revenue contribution by segment — the number that matters to the client

In [5]:
segment_summary = (
    rfm.groupby("Segment")
       .agg(Customers=("CustomerID", "count"), Revenue=("Monetary", "sum"))
       .assign(Revenue_share=lambda x: (x["Revenue"] / x["Revenue"].sum() * 100).round(1))
       .sort_values("Revenue", ascending=False)
)
segment_summary

,Customers,Revenue,Revenue_share
Segment,,,
Champions,1300,1.212812e+07,68.4
Loyal Customers,1403,2.713961e+06,15.3
At Risk,824,1.633975e+06,9.2
Lost,1523,6.671219e+05,3.8
New Customers,443,3.946386e+05,2.2
Needs Attention,385,2.056168e+05,1.2


In [8]:
out_path = PROCESSED_DIR / "rfm_segments.csv"
rfm.to_csv(out_path, index=False)

## 5. Save

In [6]:
out_path = PROCESSED_DIR / "rfm_segments.csv"
rfm.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(rfm):,} customers)")

Saved: ..\data\processed\rfm_segments.csv  (5,878 customers)


## Summary

- Scored every customer on Recency, Frequency, and Monetary value.
- Mapped scores to six actionable segments (Champions, Loyal Customers, New Customers, At Risk, Needs Attention, Lost).
- **[Fill in once run]**: Segment "X" represents **[Y%]** of customers but drives **[Z%]** of revenue — the concrete number that anchors the executive summary's recommendations.
- Next: statistical validation in `r/04_statistical_validation.R`, then `03_cohort_analysis.ipynb`.

In [9]:
import os
os.listdir("../data/processed")

['.gitkeep', 'cohort_table.csv', 'rfm_segments.csv', 'transactions_clean.csv']